# 03 - MF-BPR: Matrix Factorization with Bayesian Personalized Ranking

From scratch in PyTorch (project requirement - no RecBole, Surprise, or LensKit).

## Why BPR and not rating prediction?

The task is top-K ranking from positive interactions, not star-rating prediction. Pointwise methods such as WMF and FunkSVD push each observation toward a target value. BPR instead learns that, for a user $u$, an interacted item $i$ should rank above a sampled item $j$.

## The model

Score = dot product of latent factors, plus an item bias:

$$\hat{x}_{ui} = p_u^\top q_i + b_i$$

- $p_u \in \mathbb{R}^d$: user factors - "what this reader likes"
- $q_i \in \mathbb{R}^d$: item factors - "what this book is"
- $b_i$: item bias - absorbs global popularity so the factors can focus on *taste*, not just "is this popular"

## The BPR loss

For a triplet $(u, i, j)$ - user, positive item, sampled negative item:

$$\mathcal{L} = -\ln \sigma(\hat{x}_{ui} - \hat{x}_{uj}) + \lambda \left(\|p_u\|^2 + \|q_i\|^2 + \|q_j\|^2\right)$$

Maximizing $\sigma(\hat{x}_{ui} - \hat{x}_{uj})$ raises the probability that the model ranks $i$ above $j$ for $u$. This is a differentiable approximation to AUC. Training uses mini-batch SGD over sampled triplets.

## Negative sampling: popularity-based

Negatives are sampled in proportion to popularity: $p(j) \propto \text{count}(j)^{0.75}$. Uniform draws often select obscure books that are already easy to rank below the positive item. Popular unread books are harder comparisons. The 0.75 exponent keeps the distribution from concentrating only on bestsellers.

## 0. Setup

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

# SMOKE MODE: quick end-to-end plumbing check (e.g., on a laptop before real GPU training).
# Trains 1 config for fewer epochs. Artifacts are exported and valid but WEAK -
# never report SMOKE numbers; rerun with SMOKE = False for the real results.
SMOKE = False

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/recsys_kindle/data'
except ImportError:
    DATA_DIR = './data'

train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
val   = pd.read_parquet(os.path.join(DATA_DIR, 'val.parquet'))
test  = pd.read_parquet(os.path.join(DATA_DIR, 'test.parquet'))
with open(os.path.join(DATA_DIR, 'id_mappings.json')) as f:
    mappings = json.load(f)
N_USERS, N_ITEMS = len(mappings['user2idx']), len(mappings['item2idx'])
print(f'{N_USERS:,} users x {N_ITEMS:,} items')

Device: mps


127,585 users x 120,573 items


In [2]:
# Training positives only (rating >= 4, see notebook 01)
pos = train[train.is_positive][['user_idx', 'item_idx']].astype(int)
pos_u = pos.user_idx.to_numpy()
pos_i = pos.item_idx.to_numpy()
print(f'{len(pos):,} positive training pairs')

# For filtering negatives + seen-item exclusion at eval
user_pos_sets = pos.groupby('user_idx')['item_idx'].agg(set).to_dict()
seen_train    = train.groupby('user_idx')['item_idx'].agg(set).to_dict()

# Popularity-based negative sampling distribution: p(j) ∝ count(j)^0.75
item_counts = np.bincount(pos_i, minlength=N_ITEMS).astype(np.float64)
neg_p = item_counts ** 0.75
neg_p /= neg_p.sum()

2,039,102 positive training pairs


## 1. Evaluation helpers (same protocol as notebook 02)

Identical metrics and filtering as the baselines - that's what makes the final comparison table valid.

In [3]:
def build_truth(part):
    p = part[part.is_positive & part.user_idx.notna() & part.item_idx.notna()]
    return p.groupby('user_idx')['item_idx'].agg(set).to_dict()

truth_val, truth_test = build_truth(val), build_truth(test)

def recall_at_k(recs, truth, k):
    s = [len(set(recs.get(u, [])[:k]) & rel) / len(rel) for u, rel in truth.items() if rel]
    return float(np.mean(s)) if s else 0.0

def ndcg_at_k(recs, truth, k):
    scores = []
    for u, rel in truth.items():
        top = recs.get(u, [])[:k]
        dcg = sum(1.0 / np.log2(i + 2) for i, it in enumerate(top) if it in rel)
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(rel), k)))
        scores.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(scores))

def catalog_coverage(recs, n_items, k):
    seen = set()
    for r in recs.values():
        seen.update(r[:k])
    return len(seen) / n_items

print(f'Evaluable users: val {len(truth_val):,} | test {len(truth_test):,}')

Evaluable users: val 45,255 | test 34,191


## 2. The model

In [4]:
class MFBPR(nn.Module):
    def __init__(self, n_users, n_items, dim=64):
        super().__init__()
        self.P = nn.Embedding(n_users, dim)   # user factors
        self.Q = nn.Embedding(n_items, dim)   # item factors
        self.b = nn.Embedding(n_items, 1)     # item bias
        # Tiny factors leave SGD near the ln(2) plateau.
        # A larger initialization gives the factor updates a useful scale.
        nn.init.normal_(self.P.weight, std=0.1)
        nn.init.normal_(self.Q.weight, std=0.1)
        nn.init.zeros_(self.b.weight)

    def score(self, u, i):
        return (self.P(u) * self.Q(i)).sum(-1) + self.b(i).squeeze(-1)

    def bpr_loss(self, u, i, j, reg):
        x_ui, x_uj = self.score(u, i), self.score(u, j)
        # softplus(-(x_ui - x_uj)) is stable -log(sigmoid(x_ui - x_uj)).
        # Sum reduction keeps the SGD update from shrinking with batch size.
        loss = nn.functional.softplus(-(x_ui - x_uj)).sum()
        l2 = (self.P(u).pow(2).sum(-1) + self.Q(i).pow(2).sum(-1) + self.Q(j).pow(2).sum(-1)).sum()
        return loss + reg * l2

## Defense notes
`softplus(-x)` is the stable form of `-log(sigmoid(x))`. L2 applies only to embeddings touched by the batch, so rarely updated users and items are not repeatedly penalized. A user bias cancels in $x_{ui} - x_{uj}$, so only item biases are included.

## 3. Training loop - mini-batch SGD over sampled triplets

Each epoch shuffles all positive pairs and samples one negative item for each pair. The default is popularity-smoothed sampling, but the sampler also supports uniform draws for the comparison below. If a draw is a true positive for that user, it is resampled once.

In [5]:
SAMPLING = 'popularity'  # or 'uniform'

def sample_negatives(users, sampling=SAMPLING, sampler_rng=None):
    if sampling not in {'uniform', 'popularity'}:
        raise ValueError("sampling must be 'uniform' or 'popularity'")
    sampler_rng = rng if sampler_rng is None else sampler_rng
    p = neg_p if sampling == 'popularity' else None
    j = sampler_rng.choice(N_ITEMS, size=len(users), p=p)
    # one resampling pass for collisions (j is actually a positive of u)
    bad = np.fromiter((j[k] in user_pos_sets.get(users[k], ()) for k in range(len(users))),
                      dtype=bool, count=len(users))
    if bad.any():
        j[bad] = sampler_rng.choice(N_ITEMS, size=bad.sum(), p=p)
    return j

def train_mf_bpr(dim=64, lr=0.05, reg=1e-5, epochs=15, batch_size=8192, eval_every=3,
                 sampling=SAMPLING, seed=42):
    torch.manual_seed(seed)
    local_rng = np.random.default_rng(seed)
    model = MFBPR(N_USERS, N_ITEMS, dim).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    n = len(pos_u)
    history = []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        perm = local_rng.permutation(n)
        u_all, i_all = pos_u[perm], pos_i[perm]
        j_all = sample_negatives(u_all, sampling=sampling, sampler_rng=local_rng)
        total = 0.0
        model.train()
        for s in range(0, n, batch_size):
            u = torch.as_tensor(u_all[s:s+batch_size], device=device)
            i = torch.as_tensor(i_all[s:s+batch_size], device=device)
            j = torch.as_tensor(j_all[s:s+batch_size], device=device)
            loss = model.bpr_loss(u, i, j, reg)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()   # loss is a batch SUM now; total/n below = per-triplet average
        msg = f'epoch {epoch:2d} | loss {total/n:.4f} | {time.time()-t0:.0f}s'
        if epoch % eval_every == 0 or epoch == epochs:
            r20 = quick_recall(model, truth_val, k=20, sample=20000)
            history.append((epoch, total/n, r20))
            msg += f' | val Recall@20 {r20:.4f}'
        print(msg)
    return model, history

## 4. Retrieval evaluation with FAISS

The project indexes item embeddings with FAISS and retrieves the top 100 candidates per user.

FAISS uses inner-product search, while the model score is $p_u^\top q_i + b_i$. Appending the bias to each item vector and 1 to each user vector gives the same score:
$[q_i; b_i] \cdot [p_u; 1] = p_u^\top q_i + b_i$. Exact same scores, pure dot product.

In [6]:
# faiss-cpu is sufficient for this exact index.
import faiss

def build_faiss(model):
    Q = model.Q.weight.detach().cpu().numpy().astype('float32')
    b = model.b.weight.detach().cpu().numpy().astype('float32')
    item_vecs = np.hstack([Q, b])                      # [q_i ; b_i]
    index = faiss.IndexFlatIP(item_vecs.shape[1])      # exact inner-product search
    index.add(item_vecs)
    return index

def retrieve(model, index, users, k_retrieve=100, batch=4096):
    """FAISS top-k_retrieve per user, seen-train items filtered out."""
    users = list(users)
    recs = {}
    P = model.P.weight.detach().cpu().numpy().astype('float32')
    for s in range(0, len(users), batch):
        chunk = users[s:s+batch]
        uvecs = np.hstack([P[chunk], np.ones((len(chunk), 1), dtype='float32')])  # [p_u ; 1]
        # Retrieve extra items because seen training items are removed below.
        _, I = index.search(uvecs, k_retrieve + 50)
        for u, row in zip(chunk, I):
            seen = seen_train.get(u, set())
            recs[u] = [int(i) for i in row if i not in seen][:k_retrieve]
    return recs

def quick_recall(model, truth, k=20, sample=None):
    users = list(truth.keys())
    if sample and len(users) > sample:
        users = list(rng.choice(users, size=sample, replace=False))
    index = build_faiss(model)
    recs = retrieve(model, index, users, k_retrieve=k)
    return recall_at_k(recs, {u: truth[u] for u in users}, k)

## 5. Train + tune on validation

Hyperparameters are selected on validation data only; the test set remains untouched until the next section. The grid varies factor dimension and learning rate while keeping regularization fixed.

In [7]:
faiss.omp_set_num_threads(1)
# The grid checks whether the learning rate can leave the ln(2) plateau.
CONFIGS = [(64, 0.05, 1e-5)] if SMOKE else \
          [(64, 0.05, 1e-5), (64, 0.1, 1e-5), (64, 0.2, 1e-5), (128, 0.1, 1e-5)]
EPOCHS = 3 if SMOKE else 20

RESULTS_GRID = []
for dim, lr, reg in CONFIGS:
    print(f'\n=== dim={dim} lr={lr} reg={reg} ===')
    model, hist = train_mf_bpr(dim=dim, lr=lr, reg=reg, epochs=EPOCHS, sampling=SAMPLING)
    r20 = quick_recall(model, truth_val, k=20, sample=20000)
    RESULTS_GRID.append({'dim': dim, 'lr': lr, 'reg': reg, 'val_Recall@20': r20, 'model': model})
    print(f'-> final val Recall@20: {r20:.4f}')

grid_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'model'} for r in RESULTS_GRID])
grid_df.sort_values('val_Recall@20', ascending=False)


=== dim=64 lr=0.05 reg=1e-05 ===


epoch  1 | loss 0.6886 | 13s


epoch  2 | loss 0.6645 | 11s


epoch  3 | loss 0.6163 | 15s | val Recall@20 0.0088


epoch  4 | loss 0.5266 | 17s


...

epoch 18 | loss 0.0214 | 6s | val Recall@20 0.0340


epoch 19 | loss 0.0200 | 5s


epoch 20 | loss 0.0190 | 6s | val Recall@20 0.0352


-> final val Recall@20: 0.0347


,dim,lr,reg,val_Recall@20
3,128,0.10,0.00001,0.034728
1,64,0.10,0.00001,0.033086
2,64,0.20,0.00001,0.030506
0,64,0.05,0.00001,0.029961


In [8]:
best = max(RESULTS_GRID, key=lambda r: r['val_Recall@20'])
model = best['model']
print(f"Best config: dim={best['dim']} lr={best['lr']} reg={best['reg']} (val Recall@20={best['val_Recall@20']:.4f})")

Best config: dim=128 lr=0.1 reg=1e-05 (val Recall@20=0.0347)


## 6. Final test evaluation (top-100 retrieval, metrics on top-K)

In [9]:
index = build_faiss(model)
t0 = time.time()
recs_test = retrieve(model, index, truth_test.keys(), k_retrieve=100)
print(f'Retrieved top-100 for {len(recs_test):,} users in {time.time()-t0:.0f}s')

row_mf = {'Model': 'MF-BPR',
          'Recall@20': recall_at_k(recs_test, truth_test, 20),
          'Recall@50': recall_at_k(recs_test, truth_test, 50),
          'NDCG@10':   ndcg_at_k(recs_test, truth_test, 10),
          'Coverage@10': catalog_coverage(recs_test, N_ITEMS, 10)}

with open(os.path.join(DATA_DIR, 'baseline_results.json')) as f:
    rows = json.load(f)
rows = [r for r in rows if r['Model'] != 'MF-BPR'] + [row_mf]
with open(os.path.join(DATA_DIR, 'baseline_results.json'), 'w') as f:
    json.dump(rows, f, indent=2)

pd.DataFrame(rows).set_index('Model').round(4)

Retrieved top-100 for 34,191 users in 5s


,Recall@20,Recall@50,NDCG@10,Coverage@10
Model,,,,
Random,0.0001,0.0004,0.0000,0.9415
Popularity,0.0141,0.0242,0.0053,0.0002
Two-tower,0.0572,0.0967,0.0215,0.2996
Two-tower + LightGBM,0.0667,0.1072,0.0278,0.2885
MF-BPR,0.0263,0.0472,0.0098,0.2219


The 128-dimensional model with learning rate 0.1 gave the best validation Recall@20 (0.0347). The other settings ranged from 0.0300 to 0.0331; the low learning rate trained more slowly, while 0.2 peaked earlier and then declined. On test, MF-BPR reached Recall@20 0.0263 and NDCG@10 0.0098, ahead of popularity but below the two-tower model.

With a tiny initialization and mean-reduced loss, BPR can stay near ln(2): positive and negative scores are almost equal, so SGD barely moves the factors. Initializing at std 0.1 and summing the batch loss gave the factors a usable update.

## 7. Negative-sampling comparison

Both runs use the validation-selected configuration and the same seed. Only the negative-sampling distribution changes.

In [10]:
sampling_rows = []
for sampling in ['uniform', 'popularity']:
    print(f'\n=== sampling={sampling} ===')
    sampled_model, _ = train_mf_bpr(
        dim=best['dim'], lr=best['lr'], reg=best['reg'], epochs=EPOCHS,
        sampling=sampling, seed=42
    )
    sampled_index = build_faiss(sampled_model)
    sampled_recs = retrieve(sampled_model, sampled_index, truth_test.keys(), k_retrieve=100)
    sampling_rows.append({
        'Sampling': sampling,
        'Recall@20': recall_at_k(sampled_recs, truth_test, 20),
        'Recall@50': recall_at_k(sampled_recs, truth_test, 50),
        'NDCG@10': ndcg_at_k(sampled_recs, truth_test, 10),
        'Coverage@10': catalog_coverage(sampled_recs, N_ITEMS, 10),
    })
sampling_df = pd.DataFrame(sampling_rows).set_index('Sampling')
sampling_df.round(4)


=== sampling=uniform ===


epoch  1 | loss 0.5637 | 6s


epoch  2 | loss 0.4391 | 6s


epoch  3 | loss 0.3319 | 7s | val Recall@20 0.0103


epoch  4 | loss 0.2405 | 5s


...

epoch 17 | loss 0.0231 | 11s


epoch 18 | loss 0.0214 | 9s | val Recall@20 0.0359


epoch 19 | loss 0.0200 | 9s


epoch 20 | loss 0.0190 | 10s | val Recall@20 0.0354


,Recall@20,Recall@50,NDCG@10,Coverage@10
Sampling,,,,
uniform,0.0206,0.0418,0.0076,0.0737
popularity,0.0263,0.0472,0.0098,0.2219


Popularity-smoothed negatives beat uniform negatives on every test metric. Recall@20 rose from 0.0206 to 0.0263, NDCG@10 from 0.0076 to 0.0098, and coverage from 0.0737 to 0.2219. Sampling popular unread items gave the model more useful ranking comparisons than sampling mostly obscure books.

## 8. Export artifacts

In [11]:
ART = os.path.join(DATA_DIR, 'artifacts_mf_bpr')
os.makedirs(ART, exist_ok=True)
np.save(os.path.join(ART, 'user_factors.npy'), model.P.weight.detach().cpu().numpy())
np.save(os.path.join(ART, 'item_factors.npy'), model.Q.weight.detach().cpu().numpy())
np.save(os.path.join(ART, 'item_bias.npy'),   model.b.weight.detach().cpu().numpy())
torch.save(model.state_dict(), os.path.join(ART, 'mf_bpr.pt'))
faiss.write_index(index, os.path.join(ART, 'mf_bpr_faiss.index'))
print('Saved factors, state dict and FAISS index to', ART)

Saved factors, state dict and FAISS index to ./data/artifacts_mf_bpr
